<a href="https://colab.research.google.com/github/Nihat0311/Deep-Learning/blob/main/DL_task_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
model=nn.Linear(in_features=3,out_features=1)
def forward(self,x):
    z=self.model(x)
    return z

In [ ]:
class MultiInputModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(3 + 2, 1)
        self.relu=nn.ReLU()

    def forward(self, input_a, input_b):
        combined = torch.cat((input_a, input_b), dim=1)
        output = self.linear(combined)
        return output
model = MultiInputModel()

input_a = torch.randn(1, 3)
input_b = torch.randn(1, 2)

prediction = model(input_a, input_b)

In [ ]:
class DeepMultiInputModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden1 = nn.Linear(5, 10)
        self.hidden2 = nn.Linear(10, 5)
        self.output_layer = nn.Linear(5, 1)
        self.relu = nn.ReLU()

    def forward(self, input_a, input_b):
        combined = torch.cat((input_a, input_b), dim=1)
        x = self.hidden1(combined)
        x = self.relu(x)
        x = self.hidden2(x)
        x = self.relu(x)
        out = self.output_layer(x)
        return out
model = DeepMultiInputModel()

input_a = torch.randn(1, 3)
input_b = torch.randn(1, 2)

prediction = model(input_a, input_b)

In [ ]:
class WideAndDeepModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.deep_layer1 = nn.Linear(5, 10)
        self.deep_layer2 = nn.Linear(10, 5)
        self.relu = nn.ReLU()
        self.final_output = nn.Linear(5 + 5, 1)

    def forward(self, input_a, input_b):
        wide_part = torch.cat((input_a, input_b), dim=1)

        deep = self.deep_layer1(wide_part)
        deep = self.relu(deep)
        deep = self.deep_layer2(deep)
        deep = self.relu(deep)
        combined_all = torch.cat((wide_part, deep), dim=1)

        out = self.final_output(combined_all)
        return out

model = WideAndDeepModel()
in_a = torch.randn(1, 3)
in_b = torch.randn(1, 2)

output = model(in_a, in_b)

In [ ]:
import torch.optim as optim
class SimpleWideDeep(nn.Module):
    def __init__(self):
        super().__init__()
        self.deep_layers = nn.Sequential(
            nn.Linear(2, 5),
            nn.ReLU(),
            nn.Linear(5, 2),
            nn.ReLU()
        )
        self.output_layer = nn.Linear(2 + 2, 1)

    def forward(self, x):
        wide = x
        deep = self.deep_layers(x)
        combined = torch.cat((wide, deep), dim=1)
        return self.output_layer(combined)

torch.manual_seed(42)
X = torch.randn(100, 2)
y = 2 * X[:, 0:1] + 3 * X[:, 1:2]

model = SimpleWideDeep()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

epochs = 100
for epoch in range(epochs):
    predictions = model(X)
    loss = criterion(predictions, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

test_input = torch.tensor([[1.0, 2.0]])

Epoch [20/100], Loss: 0.0074
Epoch [40/100], Loss: 0.0038
Epoch [60/100], Loss: 0.0027
Epoch [80/100], Loss: 0.0020
Epoch [100/100], Loss: 0.0015


In [ ]:
class NonLinearWideDeep(nn.Module):
    def __init__(self):
        super().__init__()
        self.deep_layers = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 4)
        )
        self.output_layer = nn.Linear(2 + 4, 1)

    def forward(self, x):
        wide = x
        deep = self.deep_layers(x)
        combined = torch.cat((wide, deep), dim=1)
        return self.output_layer(combined)

torch.manual_seed(42)
X = torch.randn(1000, 2)
y = X[:, 0:1]**2 + X[:, 1:2]

model = NonLinearWideDeep()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 500
for epoch in range(epochs):
    predictions = model(X)
    loss = criterion(predictions, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

test_input = torch.tensor([[3.0, 1.0]])
model.eval()
with torch.no_grad():
    res = model(test_input)
    print(f"\nTest [3.0, 1.0] üçün nəticə: {res.item():.4f} (Hədəf: 10.0)")

Epoch [100/500], Loss: 0.0562
Epoch [200/500], Loss: 0.0069
Epoch [300/500], Loss: 0.0042
Epoch [400/500], Loss: 0.0029
Epoch [500/500], Loss: 0.0023

Test [3.0, 1.0] üçün nəticə: 10.1934 (Hədəf: 10.0)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class RegularizedModel(nn.Module):
    def __init__(self, use_dropout=False):
        super().__init__()

        self.use_dropout = use_dropout
        self.layer1 = nn.Linear(2, 64)
        self.dropout = nn.Dropout(p=0.3)
        self.layer2 = nn.Linear(64, 32)
        self.layer3 = nn.Linear(32, 4)
        self.relu = nn.ReLU()
        self.output_layer = nn.Linear(2 + 4, 1)

    def forward(self, x):
        wide = x
        x_d = self.relu(self.layer1(x))
        if self.use_dropout:
            x_d = self.dropout(x_d)
        x_d = self.relu(self.layer2(x_d))
        x_d = self.relu(self.layer3(x_d))
        combined = torch.cat((wide, x_d), dim=1)
        return self.output_layer(combined)
X = torch.randn(200, 2)
y = X[:, 0:1]**2 + X[:, 1:2]

def train_and_get_loss(model, X, y, epochs=200):
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        pred = model(X)
        loss = criterion(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return loss.item()

model_no_dropout = RegularizedModel(use_dropout=False)
model_with_dropout = RegularizedModel(use_dropout=True)

loss_no = train_and_get_loss(model_no_dropout, X, y)
loss_with = train_and_get_loss(model_with_dropout, X, y)

print(f"Dropout-suz son Training Loss: {loss_no:.6f}")
print(f"Dropout ilə son Training Loss: {loss_with:.6f}")

Dropout-suz son Training Loss: 0.009937
Dropout ilə son Training Loss: 0.125431


In [ ]:
class RecommendationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.deep_path = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, 8),
            nn.ReLU()
        )
        self.output_layer = nn.Linear(4 + 8, 1)

    def forward(self, user_features, item_features):
        raw_input = torch.cat((user_features, item_features), dim=1)
        deep_out = self.deep_path(raw_input)
        combined = torch.cat((raw_input, deep_out), dim=1)
        final_score = self.output_layer(combined)
        return final_score
torch.manual_seed(42)
X_user = torch.randn(500, 2)
X_item = torch.randn(500, 2)
y = (X_user[:, 0:1] * 0.5) + (X_item[:, 1:2] * 2.0)

model = RecommendationModel()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    model.train()
    preds = model(X_user, X_item)
    loss = criterion(preds, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

new_user = torch.tensor([[25.0, 85.0]])
new_item = torch.tensor([[150.0, 4.8]])

model.eval()
with torch.no_grad():
    prediction = model(new_user, new_item)
    print(f"\nİstifadəçi üçün təxmin edilən score: {prediction.item():.2f}")

Epoch 20, Loss: 1.2789
Epoch 40, Loss: 0.4093
Epoch 60, Loss: 0.2150
Epoch 80, Loss: 0.1786
Epoch 100, Loss: 0.1323

İstifadəçi üçün təxmin edilən score: 6.30
